# Pipeline 06: Synthetic Data Distillation

**Goal:** Generate a high-quality, large-scale instruction dataset to fine-tune a Small Language Model (SLM) like Qwen 2.5 or Llama 3.

**Architecture:**
Instead of manually typing hundreds of prompts into a UI, this pipeline utilizes "Knowledge Distillation." We provide a "Teacher" model (Claude) with a few human-written seed examples. We then programmatically command the Teacher to generate hundreds of highly diverse permutations of these prompts, pairing each with the exact mathematical JSON schema required by our rendering engine. This instantly builds a robust `.jsonl` dataset ready for QLoRA fine-tuning.

In [6]:
# --- 1. Setup and Distillation Engine ---
from google.colab import drive
drive.mount('/content/drive')

!pip install anthropic tqdm -q

import os
import re
import json
import random
from tqdm.notebook import tqdm
from google.colab import userdata
from anthropic import Anthropic

# Set up our project folders
PROJECT = '/content/drive/MyDrive/photo-style-rl'
DATASET_PATH = f'{PROJECT}/data/slm_training_dataset.jsonl'
os.makedirs(os.path.dirname(DATASET_PATH), exist_ok=True)

# Initialize Claude
client = Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# The DNA: Your natural editing instructions
seed_prompts = [
    "brighten the subjects and make their skin tones warmer",
    "make the sunset more dramatic and lift the shadows so it's not too contrasty",
    "make this feel more like a vintage film aesthetic of an old Asian street",
    "reduce the highlights and raise the shadows while overall brightening up the subject's face",
    "add some noise and give this have a cinematic movie haze",
    "i want this photo to look as though i took it in a fujifilm x100vi"
    "change the orange to be more faded and make the sky more teal"
    "make this in the style of director Wong-Kar Wai's film aesthetic"
    "change the vibe to a late night cyberpunk neon street scene"
    "reduce the contrast on the person's face so the impurities don't stand out"
    "the top left corner is too bright. reduce it slightly and change the green hue to be more olive-ish"
    "selectively apply some radial brushes around the subject and reduce the brightness of everything else"
    "you are an expert photographer and editor. edit this in my style in what you think would be the best"
    "i want the sunset to be more vibrant and a bit more pink to represent dusk"
    "can you make this feel more punchy and powerful while still feeling tasteful?"
    "mimic a real-life documentary photo of a Pulitzer-worthy moment"
]

available_regions = [
    "global", "subject", "background", "sky", "ground",
    "foliage", "water", "building", "clothing", "face"
]

def generate_synthetic_batch(batch_size=10):
    """Commands the Teacher model to generate new training pairs."""
    sys_prompt = f"""You are an expert photo editor generating synthetic training data.
    Generate {batch_size} BRAND NEW, highly diverse photo editing prompts inspired by the human seeds.
    Vary the tone (casual, technical, abstract) and the requested edits.

    For each prompt, provide the correct JSON math required to achieve the look.
    Available regions: {available_regions}

    Output ONLY a valid JSON array of objects. Do not include markdown or explanations.

    Required Schema:
    [
      {{
        "user_prompt": "make it look cold and moody",
        "json_payload": {{
            "global": {{"tone_curve": {{"shadows": -15.0}}}},
            "sky": {{"color_mixer": {{"blues": {{"h": -10.0, "s": -20.0}}}}}}
        }}
      }}
    ]
    """

    seeds_text = "\n".join([f"- {p}" for p in seed_prompts])
    user_message = f"Here are the seed examples. Generate {batch_size} new pairs now:\n{seeds_text}"

    try:
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=2500,
            system=sys_prompt,
            messages=[{"role": "user", "content": user_message}]
        )

        # Bulletproof JSON extraction using Regular Expressions (ignores markdown bugs)
        text = response.content[0].text
        json_match = re.search(r'\[.*\]', text, re.DOTALL)

        if json_match:
            return json.loads(json_match.group(0))
        else:
            print("Failed to find a JSON array in the response.")
            return []

    except Exception as e:
        print(f"API or Parsing Error: {e}")
        return []

print(f"Distillation Engine initialized. Saving to: {DATASET_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Distillation Engine initialized. Saving to: /content/drive/MyDrive/photo-style-rl/data/slm_training_dataset.jsonl


## Batch Execution & Data Verification

The pipeline executes the synthetic generation in small batches to ensure output quality and prevent API timeouts. Each successfully generated pair is instantly formatted for instruction fine-tuning and appended to the dataset file. Finally, a random sample is pulled from the generated dataset to visually verify the structural integrity of the output.

In [7]:
# --- 2. Run the Engine and Verify Data ---
TOTAL_BATCHES = 5

print(f"Starting generation of {TOTAL_BATCHES * 10} synthetic training pairs...\n")

successful_pairs = 0

# Open in append mode to protect existing data
with open(DATASET_PATH, 'a') as f:
    for i in tqdm(range(TOTAL_BATCHES), desc="Distilling Data"):

        batch_data = generate_synthetic_batch(batch_size=10)

        if not batch_data:
            continue

        for item in batch_data:
            # Format exactly as needed for SLM instruction tuning
            training_record = {
                "messages": [
                    {"role": "user", "content": item.get("user_prompt", "")},
                    {"role": "assistant", "content": json.dumps(item.get("json_payload", {}))}
                ]
            }

            f.write(json.dumps(training_record) + "\n")
            successful_pairs += 1

print(f"\n✅ Generation complete! Added {successful_pairs} new records.")

# --- Sanity Check ---
print("\n--- Dataset Verification ---")
if os.path.exists(DATASET_PATH):
    with open(DATASET_PATH, 'r') as f:
        lines = f.readlines()

    print(f"Total training records ready for SLM: {len(lines)}\n")

    if len(lines) > 0:
        random_record = json.loads(random.choice(lines))
        print("Sample Input:")
        print(f"👉 {random_record['messages'][0]['content']}")
        print("\nSample Output Target:")
        parsed_math = json.loads(random_record['messages'][1]['content'])
        print(json.dumps(parsed_math, indent=2))

Starting generation of 50 synthetic training pairs...



Distilling Data:   0%|          | 0/5 [00:00<?, ?it/s]


✅ Generation complete! Added 50 new records.

--- Dataset Verification ---
Total training records ready for SLM: 50

Sample Input:
👉 create that golden hour magic but the subject's face is getting lost in shadow

Sample Output Target:
{
  "face": {
    "shadows": 30.0,
    "highlights": -5.0
  },
  "sky": {
    "color_mixer": {
      "yellows": {
        "s": 25.0
      },
      "oranges": {
        "s": 20.0
      }
    }
  },
  "global": {
    "warmth": 15.0
  }
}
